# RSNA Knee explicit sharded fallback: merge-train

All-seven metadata and global longest-series selection are validated before local archive filtering. A shard produces explicitly partial frozen features; it is never called full training. Merge requires 4,407 studies, 21,334 selected slot-series and 85,336 slices before fresh heads train. Same 224px/.92 FOV/K4/author_no_rope extraction, fixed 12 epochs and Gold58 exclusion. Private medical tables are read from the small attached metadata asset, never embedded here.

In [ ]:
import time
BOOTSTRAP_MONOTONIC=time.monotonic()
print('{"event":"bootstrap_start","diagnostic_schema":1}',flush=True)
import json,os,time,traceback
BOOTSTRAP_DIR='/kaggle/working'
BOOTSTRAP_STAGE='notebook_start'
os.makedirs(BOOTSTRAP_DIR,exist_ok=True)
def bootstrap_checkpoint(stage):
    global BOOTSTRAP_STAGE
    BOOTSTRAP_STAGE=stage
    event={'diagnostic_schema':1,'event':'bootstrap_checkpoint','stage':stage,'unix_seconds':time.time()}
    with open(os.path.join(BOOTSTRAP_DIR,'bootstrap_progress.json'),'w') as stream:
        json.dump(event,stream,indent=2)
    with open(os.path.join(BOOTSTRAP_DIR,'bootstrap_events.jsonl'),'a') as stream:
        stream.write(json.dumps(event)+'\n')
    print(json.dumps(event),flush=True)
def bootstrap_save_failure(error):
    event={'diagnostic_schema':1,'event':'bootstrap_failure','stage':BOOTSTRAP_STAGE,
           'error_type':type(error).__name__,'error':str(error),'traceback':traceback.format_exc(),
           'unix_seconds':time.time()}
    try:
        with open(os.path.join(BOOTSTRAP_DIR,'bootstrap_failure.json'),'w') as stream:
            json.dump(event,stream,indent=2)
    except BaseException as save_error:
        print(json.dumps({'event':'bootstrap_failure_save_error','error_type':type(save_error).__name__}),flush=True)
    print(json.dumps(event),flush=True)
bootstrap_checkpoint('notebook_start')
with open(os.path.join(BOOTSTRAP_DIR,'bootstrap_start.json'),'w') as stream:
    json.dump({'diagnostic_schema':1,'event':'bootstrap_start','unix_seconds':time.time()},stream,indent=2)


In [ ]:
COMPRESSED_PAYLOAD='...'
SETTINGS={'mode': 'merge-train', 'shards': [], 'config': {'experiment': 'orthofoundation_frozen_attention_v1', 'encoder': 'dinov3_vitl16', 'extraction_variant': 'author_no_rope', 'feature_dim': 1024, 'checkpoint_sha256': '385a775822107b68eaa486336feb982e1ce7bd6d4e8c03ceb482a0bf546f2ff9', 'checkpoint_bytes': 1213056638, 'resolution': 224, 'slices_per_slot': 4, 'fov': 0.92, 'image_batch_size': 16, 'workers': 2, 'pilot_studies': 16, 'max_extract_seconds': 9000, 'head_kind': 'attention', 'head_dim': 256, 'head_dropout': 0.2, 'head_epochs': 12, 'head_batch_size': 64, 'head_lr': 0.001, 'head_weight_decay': 0.01, 'weak_holdout_fold': 0, 'seed': 42}, 'global_manifest_sha256': 'b2a87794d768ca3f6ee3968345f840f1028cff004708a2b3e0cfdfcfb2c15e60', 'assets_manifest_sha256': '6feeb03c976483c2b91cca667f4fdbefa998d3620a93eb6006de22a13cbdeb4f', 'metadata_dataset': 'alanchoo/rsna-knee-ortho-global-metadata-v1', 'assets_dataset': 'alanchoo/orthofoundation-l-assets-v1', 'cache_kernel_ids': [], 'partial_kernels': ['alanchoo/rsna-knee-ortho-shard-0', 'alanchoo/rsna-knee-ortho-shard-1', 'alanchoo/rsna-knee-ortho-shard-2', 'alanchoo/rsna-knee-ortho-shard-3', 'alanchoo/rsna-knee-ortho-shard-4', 'alanchoo/rsna-knee-ortho-shard-5', 'alanchoo/rsna-knee-ortho-shard-6'], 'resume_kernel': None, 'max_seconds': 1500, 'expected_images_for_shards': 0, 'planned_extraction_jobs': 7, 'total_frozen_budget_seconds': 9000, 'head_fit_budget_seconds': 1200, 'head_job_reserve_seconds': 1500, 'source_sha256': {'orthofoundation/__init__.py': 'f3df43207f5d18497b4f698aa91c746bf81dcd4edb74b81784136acd8aa52cf1', 'orthofoundation/cache.py': '700767576eb7b6d9dfa22f77bdf2088411ba4bdf6036c80db58bf608c96f7485', 'orthofoundation/data.py': 'f3b63ab922f0452c065126dc2ef1c87442a0b988dff29d25c201666a8a5310aa', 'orthofoundation/encoder.py': '86defa5f943bfc161603c7b377bbf08a9d5c2a270d01b1df5c039ab59cff7ecb', 'orthofoundation/heads.py': 'c814b1bcbf1ac1322b717c1f1e6ba7b9173357e956e9104b2c77d0351a833f1f', 'orthofoundation/run.py': '740c2e2599516b1876e7d7226bb0f11c51205cbdaec0b6eebf44dc740c659e10', 'orthofoundation/train.py': '928121a030aabf4fc58594da967ad98ff43a7ce915474d69a909ab09652a7dc1', 'sharded_frozen_extract.py': 'ffc2d904e25f7bdbd506f8c6cd58607484ffacc0305a9665fd538778ca122441'}}
try:
    bootstrap_checkpoint('imports')
    from pathlib import Path
    import base64,hashlib,json,math,os,shutil,subprocess,sys,tarfile,time,zlib
    CODE=Path('/kaggle/working/sharded_source'); CODE.mkdir(exist_ok=True)
    bootstrap_checkpoint('vendoring')
    for name,encoded in json.loads(zlib.decompress(base64.b64decode(COMPRESSED_PAYLOAD))).items():
        destination=CODE/name; destination.parent.mkdir(exist_ok=True); destination.write_bytes(base64.b64decode(encoded))
        if hashlib.sha256(destination.read_bytes()).hexdigest()!=SETTINGS['source_sha256'][name]:
            raise RuntimeError('Vendored sharded source SHA mismatch: '+name)
    (CODE/'config.json').write_text(json.dumps(SETTINGS['config'],indent=2))
    INPUT=Path('/kaggle/input')
    def find_input(identifier,marker,kind='datasets'):
        owner,slug=identifier.split('/')
        candidates=[INPUT/kind/owner/slug,INPUT/slug,INPUT/'kernels'/owner/slug,INPUT/owner/slug]
        found=[path for path in candidates if (path/marker).is_file()]
        if not found:
            for pattern in [f'*/{slug}',f'*/*/{slug}',f'*/*/*/{slug}']:
                found.extend(path for path in INPUT.glob(pattern) if (path/marker).is_file())
        found=list(dict.fromkeys(found))
        if len(found)!=1:raise RuntimeError(f'Missing/ambiguous private source {identifier}: {found}')
        return found[0]
    def unpack(archive,target,expected,allowed_members=None):
        if archive.stat().st_size!=expected['bytes'] or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected['sha256']:
            raise RuntimeError('Private archive size/SHA differs from pin')
        target.mkdir(exist_ok=True)
        with tarfile.open(archive,'r:gz') as tar:
            members=tar.getmembers(); files=[]
            for member in members:
                path=Path(member.name)
                if path.is_absolute() or '..' in path.parts or member.issym() or member.islnk():
                    raise RuntimeError('Unsafe private archive member')
                if member.isfile():files.append(member.name)
            if len(files)!=len(set(files)) or (allowed_members is not None and set(files)!=set(allowed_members)):
                raise RuntimeError('Private archive members differ from manifest')
            tar.extractall(target,filter='data')
    bootstrap_checkpoint('global_metadata_resolution')
    metadata=find_input(SETTINGS['metadata_dataset'],'global_metadata_manifest.json')
    manifest_file=metadata/'global_metadata_manifest.json'
    if hashlib.sha256(manifest_file.read_bytes()).hexdigest()!=SETTINGS['global_manifest_sha256']:
        raise RuntimeError('Global private manifest differs from prepared pin')
    manifest=json.loads(manifest_file.read_text())
    DATA=Path('/kaggle/working/global_metadata')
    bootstrap_checkpoint('global_metadata_unpack')
    unpack(metadata/manifest['archive']['filename'],DATA,manifest['archive'],[row['path'] for row in manifest['members']])
    shutil.copyfile(manifest_file,DATA/'global_metadata_manifest.json')
    cmd=[sys.executable,str(CODE/'sharded_frozen_extract.py'),'--mode',SETTINGS['mode'],
         '--global-root',str(DATA),'--global-manifest-sha256',SETTINGS['global_manifest_sha256'],
         '--assets-manifest-sha256',SETTINGS['assets_manifest_sha256'],'--config-json',str(CODE/'config.json'),
         '--output-dir','/kaggle/working/run','--max-seconds',str(SETTINGS['max_seconds']),
         '--planned-extraction-jobs',str(SETTINGS['planned_extraction_jobs'])]
    if SETTINGS['mode'] in ('pilot','extract'):
        bootstrap_checkpoint('encoder_assets_resolution')
        assets=find_input(SETTINGS['assets_dataset'],'asset_manifest.json')
        asset_file=assets/'asset_manifest.json'
        if hashlib.sha256(asset_file.read_bytes()).hexdigest()!=SETTINGS['assets_manifest_sha256']:
            raise RuntimeError('Encoder asset manifest differs from prepared pin')
        asset_manifest=json.loads(asset_file.read_text()); dinov3=asset_manifest['dinov3_source']
        bootstrap_checkpoint('encoder_source_unpack')
        source_archive=assets/dinov3['filename']
        unpack(source_archive,CODE,{'bytes':source_archive.stat().st_size,'sha256':dinov3['sha256']})
        bootstrap_checkpoint('owned_cache_resolution')
        roots=[find_input(identifier,'cache_summary.json','notebooks') for identifier in SETTINGS['cache_kernel_ids']]
        cmd+=['--shards',*[str(shard) for shard in SETTINGS['shards']],'--cache-roots',*[str(root) for root in roots],
              '--dinov3-repo',str(CODE/dinov3['archive_root']),'--checkpoint',str(assets/asset_manifest['checkpoint']['filename']),
              '--assets-manifest',str(asset_file)]
        if SETTINGS['resume_kernel']:
            prior=find_input(SETTINGS['resume_kernel'],'run/partial_manifest.json','notebooks')
            cmd+=['--resume-partial',str(prior/'run')]
    else:
        bootstrap_checkpoint('partial_features_resolution')
        roots=[find_input(identifier,'run/partial_manifest.json','notebooks') for identifier in SETTINGS['partial_kernels']]
        cmd+=['--partial-roots',*[str(root/'run') for root in roots]]
    bootstrap_checkpoint('gpu_selection')
    import torch
    if not torch.cuda.is_available():raise RuntimeError('No CUDA GPU; refuse CPU fallback')
    free=[torch.cuda.mem_get_info(index)[0] for index in range(torch.cuda.device_count())]
    env=dict(os.environ); env['PYTHONPATH']=str(CODE)+os.pathsep+env.get('PYTHONPATH','')
    env['OMP_NUM_THREADS']='1'; env['OPENBLAS_NUM_THREADS']='1'
    env['CUDA_VISIBLE_DEVICES']=str(max(range(len(free)),key=lambda index:free[index]))
    env['RSNA_SHARDED_BOOTSTRAP_START']=str(BOOTSTRAP_MONOTONIC)
    print(json.dumps({'mode':SETTINGS['mode'],'shards':SETTINGS['shards'],'max_seconds':SETTINGS['max_seconds'],
                      'launch':cmd,'gpu_free_gb':[value/1e9 for value in free]}),flush=True)
    bootstrap_checkpoint('launch')
    remaining=SETTINGS['max_seconds']-(time.monotonic()-BOOTSTRAP_MONOTONIC)
    if remaining<=0:raise RuntimeError('Private launcher Python-stage budget exhausted before child launch')
    subprocess.run(cmd,env=env,check=True,timeout=remaining)
    bootstrap_checkpoint('report')
    name={'pilot':'pilot_report.json','extract':'extraction_report.json','merge-train':'training_report.json'}[SETTINGS['mode']]
    report=json.loads((Path('/kaggle/working/run')/name).read_text())
    if SETTINGS['mode']=='extract' and not report['extraction_complete_for_shards']:raise RuntimeError('Shard feature extraction incomplete')
    if SETTINGS['mode']=='merge-train' and not report['head_training_complete']:raise RuntimeError('Merged head training incomplete')
    print(json.dumps(report,indent=2),flush=True)
    bootstrap_checkpoint('complete')
except BaseException as bootstrap_error:
    bootstrap_save_failure(bootstrap_error)
    raise
